In [1]:
import org.springframework.ai.chat.client.ChatClient
import org.springframework.ai.chat.client.advisor.MessageChatMemoryAdvisor
import org.springframework.ai.chat.memory.InMemoryChatMemoryRepository
import org.springframework.ai.chat.memory.MessageWindowChatMemory
import org.springframework.ai.openai.OpenAiChatModel
import org.springframework.ai.openai.OpenAiChatOptions
import org.springframework.ai.openai.api.OpenAiApi
import org.springframework.ai.chat.client.entity
val apiKey = System.getenv("OPENAI_API_KEY") ?: "YOUR_OPENAI_API_KEY"

val openAiApi = OpenAiApi.builder().apiKey(apiKey).build()
val openAiChatOptions = OpenAiChatOptions.builder()
    .model(OpenAiApi.ChatModel.GPT_5_CHAT_LATEST)
    .temperature(0.8)
    .build()
val chatModel = OpenAiChatModel.builder().openAiApi(openAiApi).defaultOptions(openAiChatOptions).build()
val chatClient = ChatClient.builder(chatModel)
    .defaultAdvisors(
        MessageChatMemoryAdvisor.builder(MessageWindowChatMemory.builder().chatMemoryRepository(InMemoryChatMemoryRepository()).build()).build(),
    ).build()


# EDD: Eval Driven Development

## Step One: Target Audience
- **Goals**:
  - A chat assistant for JFall attendees and speakers that helps them quickly find accurate, up-to-date conference facts and build a personal session plan.
- **Users**
  - First-time attendee
  - Java/Kotlin developer
  - Speaker
- **Scenarios**


### Users

In [2]:
data class UserProfile(
    val userType: String,
    val id: String,
    val description: String,
    val goals: List<String>
)

In [3]:
val users = listOf(
    UserProfile(
        userType = "First-time attendee",
        id = "first_time_attendee",
        description = "New to JFall (and often the venue). Asks broad, practical questions and prefers guided suggestions backed by exact facts.",
        goals = listOf(
            "Get oriented quickly (venue, schedule, logistics).",
            "Discover suitable sessions using plain-language queries.",
            "Create a small shortlist of sessions to attend."
        )
    ),
    UserProfile(
        userType = "Java/Kotlin developer",
        id = "java_kotlin_developer",
        description = "Technical attendee optimizing for relevance and depth. Uses specific topic/level queries and compares session options.",
        goals = listOf(
            "Find highly relevant sessions by topic, technology, and skill level.",
            "Compare sessions based on content, speaker, time, and room.",
            "Build and refine a conflict-free personal schedule."
        )
    ),
    UserProfile(
        userType = "Speaker",
        id = "speaker",
        description = "Presenter focused on their own talk logistics and the overall conference flow; needs quick, precise schedule/room answers.",
        goals = listOf(
            "Confirm exact time and room details for their own session.",
            "Understand overall conference logistics and schedule boundaries.",
            "Identify related or interesting sessions to attend or recommend."
        )
    )
)

### Scenarios

In [4]:
data class Scenario(
    val scenarioName: String,
    val description: String,
    val userProfileId: String,
    val userIntent: String,
    val needs: List<String>
)

In [5]:
val scenarios = listOf(
    Scenario(
        scenarioName = "FirstTimeAttendee_OrientationAndPlan",
        description = "A first-time attendee wants to understand venue logistics and discover suitable beginner-friendly sessions to attend.",
        userProfileId = "first_time_attendee",
        userIntent = "Get oriented and decide what to attend",
        needs = listOf(
            "venue_info",
            "session_info",
            "semantic_session_search",
            "shortlist_or_booking"
        )
    ),
    Scenario(
        scenarioName = "Developer_TargetedSemanticSearchAndSchedule",
        description = "A Java/Kotlin developer searches for highly relevant technical sessions, compares options, and builds a personal schedule.",
        userProfileId = "java_kotlin_developer",
        userIntent = "Optimize for technical relevance and depth",
        needs = listOf(
            "semantic_session_search",
            "session_info",
            "shortlist_or_booking"
        )
    ),
    Scenario(
        scenarioName = "Speaker_ConfirmOwnTalkAndFillGaps",
        description = "A speaker verifies their own session logistics and plans which sessions to attend or recommend around their talk.",
        userProfileId = "speaker",
        userIntent = "Confirm talk logistics and plan around it",
        needs = listOf(
            "session_info",
            "venue_info",
            "semantic_session_search",
            "shortlist_or_booking"
        )
    )
)

### Generate Test Prompts
Combine Scenarios and Users into specific, realistic prompts that can be used to evaluate the assistant's performance in each scenario. Each prompt should reflect the user's intent and needs as defined in the scenarios.

In [6]:
data class ScenarioPrompt(
    val id: String,
    val question: String,
    val userType: String,
    val scenarioName: String
) {
    override fun toString(): String = "Question:$question\nUser: $userType\nScenario: $scenarioName"
}



In [7]:
chatClient.prompt()
    .system { it.text("You are a helpful testdata generation assistant for an eval driven development workflow: $scenarios")
    }
    .user { it.text("""
Given following user profiles:
$users
---
And following scenarios:
$scenarios
---
Generate two prompts for each user profile matching a different scenario.
It must be phrased as realistic questions that a user of that type might ask the assistant. Each prompt should reflect the user's intent and needs as defined in the scenarios.
""")
}.call().entity<List<ScenarioPrompt>>().joinToString("\n\n", prefix = "-")


-Question:Hi, it’s my first time at JFall—can you tell me where registration happens and which beginner sessions would be good to start with?
User: First-time attendee
Scenario: FirstTimeAttendee_OrientationAndPlan

Question:What’s the easiest way to get from the main entrance to the keynote hall, and can you suggest two intro-level talks about Java I could add to my schedule?
User: First-time attendee
Scenario: FirstTimeAttendee_OrientationAndPlan

Question:I’m a Kotlin developer—can you find deep-dive sessions about coroutines or JVM performance that don’t overlap?
User: Java/Kotlin developer
Scenario: Developer_TargetedSemanticSearchAndSchedule

Question:Show me advanced Java sessions comparing new language features and which ones fit together in a conflict-free schedule.
User: Java/Kotlin developer
Scenario: Developer_TargetedSemanticSearchAndSchedule

Question:Can you confirm what room and time my talk on reactive Java is scheduled for, and what sessions are happening right before

In [7]:
val prompts = listOf(
    ScenarioPrompt(
        id = "q_ft_001",
        question = "Where is JFall located, and what time should I arrive?",
        userType = "First-time attendee",
        scenarioName = "FirstTimeAttendee_OrientationAndPlan"
    ),
    ScenarioPrompt(
        id = "q_ft_003",
        question = "How much does a regular ticket cost?",
        userType = "First-time attendee",
        scenarioName = "FirstTimeAttendee_OrientationAndPlan"
    ),
    ScenarioPrompt(
        id = "q_ft_004",
        question = "Find me beginner sessions about Java performance or profiling.",
        userType = "First-time attendee",
        scenarioName = "FirstTimeAttendee_OrientationAndPlan"
    ),
    ScenarioPrompt(
        id = "q_dev_003",
        question = "Search for sessions about Spring AI, LLMs, or agentic applications.",
        userType = "Java/Kotlin developer",
        scenarioName = "Developer_TargetedSemanticSearchAndSchedule"
    ),
    ScenarioPrompt(
        id = "q_dev_004",
        question = "Compare the best 2 sessions on testing (unit/integration/contract) and recommend one for advanced devs.",
        userType = "Java/Kotlin developer",
        scenarioName = "Developer_TargetedSemanticSearchAndSchedule"
    ),
    ScenarioPrompt(
        id = "q_spk_001",
        question = "Where and when am I speaking? Please include room name and start/end time.",
        userType = "Speaker",
        scenarioName = "Speaker_ConfirmOwnTalkAndFillGaps"
    ),
    ScenarioPrompt(
        id = "q_spk_003",
        question = "Find sessions related to my talk topic so I can recommend them to attendees.",
        userType = "Speaker",
        scenarioName = "Speaker_ConfirmOwnTalkAndFillGaps"
    )
)